# Gemma 4 31B APIで赤ちゃん・ママ言葉変換

このノートブックは、Google AI Studio経由でGemma 4 31B APIを呼び出し、
文章を赤ちゃん・園児語またはママ・やさしい口調に変換します。

**必要なもの:**
- Google AI Studioで取得したAPIキー
- インターネット接続

**特徴:**
- ローカルモデル不要
- GPUなしで実行可能
- リアルタイム変換
- 150文字制限と自動再試行

## セットアップ

### 1. 依存ライブラリをインストール

In [ ]:
!pip install google-generativeai -q

### 2. APIキーを設定

下のセルを実行し、Google AI Studioから取得したAPIキーを入力してください。

APIキーの取得方法:
1. https://aistudio.google.com/app/apikey にアクセス
2. 「APIキーを作成」をクリック
3. 「新しいプロジェクトでAPIキーを作成」または既存プロジェクトを選択
4. 表示されたキーをコピー

In [ ]:
import getpass
import google.generativeai as genai

api_key = getpass.getpass("Google AI Studio APIキーを入力してください: ")
genai.configure(api_key=api_key)
print("✓ APIキーが設定されました。")

In [ ]:
# 利用可能なモデルを確認（Gemma系のモデル識別子を実測する）
print("利用可能なGemmaモデル:")
print("=" * 60)

gemma_models = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        if "gemma" in m.name.lower():
            gemma_models.append(m.name)
            print(f"  {m.name}")

if not gemma_models:
    print("  （Gemmaモデルが見つかりませんでした）")
    print("\n全モデル一覧:")
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            print(f"  {m.name}")

## 実装

In [ ]:
from typing import Literal
import json

# ===== 設定 =====
MODEL_NAME = "gemma-4-31b-it"   # Issue #15で人間監督が指定したモデル
MAX_OUTPUT_CHARS = 150
DEBUG_SHOW_PROMPT = False       # Trueにすると、実際に送るプロンプトを表示します

# ===== 指示部（Issue #15の本文をそのまま使用）=====
INSTRUCTIONS = {
    "baby": """あなたは文章の言い換え器です。
入力文の意味・事実・感情を保ったまま、「幼児退行した人が話す自然な赤ちゃん・園児語」に言い換えてください。

ルール:
- 入力への返答や助言はしない。入力文そのものを言い換える
- 原文にない情報・感情・解決策を追加しない
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- ひらがなを少し多めにし、短く幼い言い回しにする
- 「ばぶ」「おぎゃー」「でちゅ」などは多用しない
- かわいさより、元の意味が伝わることを優先する
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する""",

    "mother": """あなたは文章の言い換え器です。
入力文の意味をできるだけ保ったまま、「やさしく包み込むお母さん・ママ口調」に言い換えてください。

ルール:
- 入力への返答はしない。入力文そのものを言い換える
- 原文にない出来事・感情・解決策を追加しない
- 命令、説教、冷たい表現、マサカリ表現をやわらかくする
- 必要な助言が原文にある場合は、内容を消さず任意の提案表現へ変える
- 相手の能力や人格を否定する表現は、責めない表現へ変える
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- 「よしよし」「えらいね」などは必要な場合だけ使い、多用しない
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する""",
}

# ===== 手本（few-shot）=====
# 原文にない情報を足していない例だけを置くこと。手本がそのまま挙動になります。
EXAMPLES = {
    "baby": [
        ("明日までに資料を作らないといけない。", "あしたまでに しりょう つくらなきゃ いけないの。"),
        ("レビューで指摘が多くて心が折れそう。", "レビューで ちてき いっぱいで こころ おれちゃいそう。"),
    ],
    "mother": [
        ("なんでこんな設計にしたの。ありえない。", "どうしてこの設計にしたのかな。ちょっと気になってしまって。"),
        ("明日までに資料を作らないといけない。", "あしたまでに資料を作らないといけないのね。"),
    ],
}


def build_prompt(mode: Literal["baby", "mother"], text: str, retry: bool = False) -> str:
    """Gemmaへ渡すプロンプトを組み立てる。

    入力文は必ず末尾に置き、最後を「出力:」で終える。
    こうしないと、Gemmaが入力文をルール一覧の一部として読み、
    指示を要約し返す挙動になる（実測で確認済み）。
    """
    parts = [INSTRUCTIONS[mode]]
    if retry:
        parts.append("")
        parts.append("前回は150文字を超えました。意味を保って、必ず150文字以内へ短くしてください。")
    parts.append("")
    parts.append("例:")
    for source_text, target_text in EXAMPLES[mode]:
        parts.append(f"入力: {source_text}")
        parts.append(f"出力: {target_text}")
        parts.append("")
    parts.append(f"入力: {text}")
    parts.append("出力:")
    return "\n".join(parts)


def clean_output(raw: str) -> str:
    """モデル出力から、変換後の文章だけを取り出す。"""
    text = raw.strip()

    # 次の「入力:」を作り始めた場合は、そこで切る
    for marker in ("\n入力:", "\n入力：", "\n例:"):
        if marker in text:
            text = text.split(marker)[0]

    # 先頭に「出力:」を付けてきた場合は外す
    for prefix in ("出力:", "出力："):
        if text.startswith(prefix):
            text = text[len(prefix):]

    # Markdownコードフェンスを外す
    if text.strip().startswith("```"):
        lines = [line for line in text.strip().split("\n") if not line.startswith("```")]
        text = "\n".join(lines)

    return text.strip()


def transform_text(
    mode: Literal["baby", "mother"],
    text: str,
    retry: bool = False,
) -> str:
    """文章を指定のスタイルへ言い換える。

    Args:
        mode: 変換スタイル ("baby" または "mother")
        text: 入力文章
        retry: 150文字超過時の再試行フラグ

    Returns:
        変換後の文章

    Raises:
        ValueError: 空文字列、不正なmode、入力が長すぎる場合
        RuntimeError: API呼び出しエラー、空応答
    """
    if mode not in ("baby", "mother"):
        raise ValueError(f"modeは'baby'または'mother'である必要があります。指定: {mode}")

    if not text or not text.strip():
        raise ValueError("空文字列は受け付けません。")

    if len(text) > 500:
        raise ValueError(f"入力は500文字以内である必要があります。現在: {len(text)}文字")

    prompt = build_prompt(mode, text, retry=retry)

    if DEBUG_SHOW_PROMPT:
        print("----- 送信するプロンプト -----")
        print(prompt)
        print("----- ここまで -----")

    try:
        model = genai.GenerativeModel(MODEL_NAME)
        response = model.generate_content(
            prompt,
            generation_config={
                "temperature": 0.2,       # 変換タスクなので揺れを抑える
                "max_output_tokens": 256,
                "stop_sequences": ["\n入力:", "\n例:"],
            },
        )
    except Exception as exc:
        message = str(exc)
        if "429" in message:
            raise RuntimeError("APIレート制限に達しました。しばらく待ってから再試行してください。") from exc
        if "401" in message or "403" in message or "authentication" in message.lower():
            raise RuntimeError("認証エラー。APIキーが正しいか確認してください。") from exc
        if "404" in message:
            raise RuntimeError(f"モデル {MODEL_NAME} が見つかりません。モデル一覧セルで識別子を確認してください。") from exc
        if "timeout" in message.lower() or "deadline" in message.lower():
            raise RuntimeError("APIリクエストタイムアウト。") from exc
        raise RuntimeError(f"API呼び出しエラー: {exc}") from exc

    # 安全フィルタや打ち切りで本文が無い場合があるため、candidatesから直接取る
    if not getattr(response, "candidates", None):
        raise RuntimeError("APIが候補を返しませんでした（安全フィルタの可能性があります）。")

    candidate = response.candidates[0]
    parts = getattr(getattr(candidate, "content", None), "parts", None) or []
    raw = "".join(getattr(part, "text", "") or "" for part in parts)

    if not raw.strip():
        raise RuntimeError(
            f"APIが空の応答を返しました。finish_reason={getattr(candidate, 'finish_reason', '不明')}"
        )

    return clean_output(raw)


print(f"✓ 実装完了（モデル: {MODEL_NAME}）。以下のセルで変換を実行してください。")


## 使い方

下のセルで入力文とモード（赤ちゃん/ママ）を指定して実行してください。

In [ ]:
# 赤ちゃん・園児語への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("baby", input_text)
    print(f"赤ちゃん・園児語: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

In [ ]:
# ママ・やさしい口調への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("mother", input_text)
    print(f"ママ・やさしい口調: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

## 対話的に使う

このセルを編集して、好きな文章とモードを試してください。

In [ ]:
# ここを編集して試してください
my_text = "これはテストです。自由に編集して試してください。"
my_mode = "baby"  # "baby" または "mother"

print(f"入力: {my_text}")
print(f"モード: {my_mode}")
print(f"入力文字数: {len(my_text)}")
print("-" * 50)

try:
    output = transform_text(my_mode, my_text)
    print(f"変換結果: {output}")
    print(f"出力文字数: {len(output)}")

    if len(output) > MAX_OUTPUT_CHARS:
        print(f"\n⚠️  150文字を超えたため、再試行します...")
        output = transform_text(my_mode, my_text, retry=True)
        print(f"再試行結果: {output}")
        print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"❌ エラー: {e}")

## 複数の文章を一括処理

複数の文章をまとめて変換したい場合はこのセルを使ってください。

In [ ]:
# 変換対象の文章リスト
texts = [
    "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    "テストがパスしました。",
    "エラーが発生したので、原因を調査してください。",
]

mode = "baby"  # "baby" または "mother"

print(f"モード: {mode}")
print("=" * 60)

results = []
for i, text in enumerate(texts, 1):
    print(f"\n[{i}] 入力: {text}")
    try:
        output = transform_text(mode, text)
        print(f"    出力: {output}")
        results.append({
            "input": text,
            "output": output,
            "mode": mode,
            "success": True,
        })
    except Exception as e:
        print(f"    ❌ エラー: {e}")
        results.append({
            "input": text,
            "error": str(e),
            "mode": mode,
            "success": False,
        })

print("\n" + "=" * 60)
print(f"処理完了: {sum(1 for r in results if r['success'])}/{len(results)} 成功")

## 結果をJSONで保存

変換結果をJSONファイルとして保存できます。

In [ ]:
# 結果をJSON形式で表示
if results:
    print(json.dumps(results, ensure_ascii=False, indent=2))
else:
    print("結果がありません。上のセルで処理を実行してください。")

## 注意事項

- **APIキー:** このノートブックの実行ログには、処理した文章が記録されます。個人情報を含まない内容で試してください。
- **レート制限:** Google AI Studioの無料枠にはレート制限があります。大量の処理が必要な場合は間隔を開けてください。
- **出力:** 150文字を超える場合、自動的に1回だけ再試行します。それでも超過した場合はエラーになります。
- **品質:** APIからの出力はモデルの確率的な生成です。同じ入力でも結果が異なることがあります。

## サポート

エラーが出た場合:
1. APIキーが正しく設定されているか確認
2. インターネット接続を確認
3. 入力文の長さを確認（最大500文字）
4. レート制限に達していないか確認（429エラー）